# EarthScape Climate Agency — Notebook 07: Anomaly Detection (Isolation Forest)

### Objective:
Implement Scikit-Learn's Isolation Forest algorithm to detect multi-dimensional climate anomalies based on geographic coordinates, extreme precipitation, event duration, and severity scores.


## 1. Import ML Libraries


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")


## 2. Load & Preprocess Feature Set


In [ ]:
from utils.data_utils import clean_and_engineer_features
from ml.anomaly_model import ClimateAnomalyDetector

df = clean_and_engineer_features(pd.read_csv('../WeatherEvents_Jan2016-Dec2022.csv', nrows=150000))
print(f"Features ready for Anomaly Detection: {len(df):,} records.")


## 3. Train Isolation Forest


In [ ]:
detector = ClimateAnomalyDetector(contamination=0.04, random_state=42)
detector.fit(df)
df_scored = detector.predict(df)

total_anomalies = (df_scored['is_anomaly'] == -1).sum()
pct = (total_anomalies / len(df_scored)) * 100
print(f"Identified {total_anomalies:,} anomalies ({pct:.2f}% of dataset).")


## 4. Visualize Normal vs Anomaly Distribution


In [ ]:
plt.figure(figsize=(10, 5))
sns.scatterplot(
    data=df_scored.sample(15000, random_state=42),
    x='Precipitation(in)',
    y='DurationHours',
    hue='is_anomaly',
    palette={1: 'teal', -1: 'crimson'},
    alpha=0.6
)
plt.title('Isolation Forest: Normal (1) vs Anomalous (-1) Weather Events')
plt.xlabel('Precipitation (in)')
plt.ylabel('Duration (Hours)')
plt.show()


## 5. Anomaly Timeline


In [ ]:
plt.figure(figsize=(12, 4))
df_anom = df_scored[df_scored['is_anomaly'] == -1]
anom_timeline = df_anom.groupby(['Year', 'Month']).size()
anom_timeline.plot(kind='line', marker='s', color='darkred', linewidth=2)
plt.title('Timeline of Detected Climate Anomalies')
plt.ylabel('Anomaly Count')
plt.show()


## 6. Top States with Anomalous Climate Patterns


In [ ]:
plt.figure(figsize=(10, 4))
df_anom['State'].value_counts().head(10).plot(kind='bar', color='coral')
plt.title('Top 10 States with Highest Climate Anomalies')
plt.xlabel('State')
plt.ylabel('Detected Anomalies')
plt.show()


### Conclusion:
Isolation Forest successfully isolates extreme multi-variable climate deviations, assigning quantitative anomaly scores suitable for automated alerts and risk mapping.
